# Calc Offset v1
Load car parquet data with selected columns

In [1]:
import pandas as pd
import json
import os
from glm_model_test import load_glm_model, predict_glm
from default_value_handler import check_missing_values, apply_all_defaults

In [23]:
# Load config
with open('config.json', 'r') as f:
    config = json.load(f)

parquet_path = config['car_parquet_path']
print(f"Parquet path: {parquet_path}")
short_path = '/'.join(parquet_path.split('/')[-3:])
print(f"Parquet path: {short_path}")

vin_bsst_path = config['vin_bsst_path']
print(f"vin_bsst_path path: {vin_bsst_path}")
short_vin_bsst_path = '/'.join(vin_bsst_path.split('/')[-3:])
print(f"Parquet path: {short_vin_bsst_path}")

# Load BSST_GLM_path from config
bsst_glm_path = config['BSST_GLM_path']
print(f"BSST GLM path: {bsst_glm_path}")

# Load BSST_GLM_path from config
zip_geodensity_csv = config['zip_geodensity_csv']
print(f"zip_geodensity_csv path: {zip_geodensity_csv}")


Parquet path: /Users/Mach/dev/aps/data/2026_Dmodel_data/master_dataset_car.parquet
Parquet path: data/2026_Dmodel_data/master_dataset_car.parquet
vin_bsst_path path: /Users/Mach/dev/aps/data/2026_Dmodel_data/VIN_BSST.parquet
Parquet path: data/2026_Dmodel_data/VIN_BSST.parquet
BSST GLM path: /Users/Mach/dev/aps/data/2026_Dmodel_data/BSST_json
zip_geodensity_csv path: /Users/Mach/dev/aps/data/2026_Dmodel_data/geo_2017_2023_20260128.csv


In [ ]:
# Define columns to load
columns = [
    'cef_est_curr_mi_grp_imps', # 'odometer',
    'zip_pop_dens',#'geo_pop_density_ntile',
    'dml_year_imps', #'CALC_VEH_AGE' = 2025 - 'dml_year_imps' ,
    'vc_msrp_impa',
    'st_raw',
    'insstate',
    'dml_make_raw',
    'vin',
    'vin_date', 
    'zip'
]

# Load parquet file with selected columns
df = pd.read_parquet(parquet_path, columns=columns)
print(f"Data loaded successfully!")

Data loaded successfully!


In [4]:
# Display basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

Shape: (22705842, 10)

Columns: ['cef_est_curr_mi_grp_imps', 'zip_pop_dens', 'dml_year_imps', 'vc_msrp_impa', 'st_raw', 'insstate', 'dml_make_raw', 'vin', 'vin_date', 'zip']

Data types:
cef_est_curr_mi_grp_imps    float64
zip_pop_dens                float64
dml_year_imps               float64
vc_msrp_impa                float64
st_raw                       object
insstate                     object
dml_make_raw                 object
vin                          object
vin_date                     object
zip                          object
dtype: object


In [ ]:
df_zip_geodensity = pd.read_csv(zip_geodensity_csv)
df_zip_geodensity.head()

AttributeError: module 'pandas' has no attribute 'read'

In [ ]:
# Calculate geo_pop_density_ntile using zip, geo_pop_density, and state
# Based on logic from 20260511 Entry Level Car Value GLM.ipynb

# Step 1: Add REC_CNT column
df['REC_CNT'] = 1

# Step 2: Aggregate record counts by zip
zip_counts = df.groupby(['zip']).agg({'REC_CNT': 'count'}).reset_index()

# Step 3: Merge geo data with zip counts
geo = df_zip_geodensity[['zip', 'geo_pop_density']].copy()
geo = geo.loc[geo['geo_pop_density'].notna()]
geo = geo.merge(zip_counts, how='left')

# Step 4: Sort by geo_pop_density and calculate ntile
geo.sort_values(by='geo_pop_density', inplace=True)
geo['cumsum'] = geo['REC_CNT'].cumsum()
geo['geo_pop_density_ntile'] = round(geo['cumsum'] / geo['cumsum'].max(), 2) * 100
geo = geo[['zip', 'geo_pop_density_ntile']]

# Step 5: Merge ntile to main dataframe
df = df.merge(geo, how='left')

# Step 6: Calculate state default (average ntile per state)
gpdn_default_state = df.groupby(['st_raw']).agg({'geo_pop_density_ntile': 'mean'}).reset_index()
gpdn_default_state.rename(columns={'geo_pop_density_ntile': 'geo_pop_density_ntile_state_default'}, inplace=True)
gpdn_default_state['geo_pop_density_ntile_state_default'] = round(gpdn_default_state['geo_pop_density_ntile_state_default'], 0)

# Step 7: Merge state default and fill missing values
df = df.merge(gpdn_default_state, how='left')
df['geo_pop_density_ntile'] = df['geo_pop_density_ntile'].fillna(df['geo_pop_density_ntile_state_default']).fillna(50)
del df['geo_pop_density_ntile_state_default']
del df['REC_CNT']

print(f"geo_pop_density_ntile calculated for {len(df):,} records")
print(f"geo_pop_density_ntile range: {df['geo_pop_density_ntile'].min()} to {df['geo_pop_density_ntile'].max()}")
print(f"Missing values: {df['geo_pop_density_ntile'].isna().sum()}")

In [ ]:
plt.hist(zip_pop_dens)

In [5]:
# Preview data
df.head()

,cef_est_curr_mi_grp_imps,zip_pop_dens,dml_year_imps,vc_msrp_impa,st_raw,insstate,dml_make_raw,vin,vin_date,zip
0,0.0,5156.602537,2005.0,19594.0,OH,OH,HYUNDAI,KMHWF35H45A178516,KMHWF35H45A178516_04-26-2022,43613
1,210000.0,4747.258202,2005.0,19594.0,TX,TX,HYUNDAI,KMHWF35H75A128502,KMHWF35H75A128502_11-21-2020,78758
2,70000.0,400.767060,2013.0,19545.0,CA,CA,HYUNDAI,KMHD35LE5DU068088,KMHD35LE5DU068088_05-03-2020,92315
3,60000.0,1161.108842,2013.0,18545.0,GA,GA,HYUNDAI,KMHD35LE5DU047158,KMHD35LE5DU047158_05-30-2021,30519
4,0.0,2918.222119,2013.0,18545.0,NJ,NJ,HYUNDAI,KMHD35LE5DU082332,KMHD35LE5DU082332_11-17-2019,8075


In [6]:
# Count records where cef_est_curr_mi_grp_imps = 0
zero_count = (df['cef_est_curr_mi_grp_imps'] == 0).sum()
total_count = len(df)
print(f"Records with cef_est_curr_mi_grp_imps = 0: {zero_count:,}")
print(f"Total records: {total_count:,}")
print(f"Percentage: {zero_count/total_count*100:.2f}%")



Records with cef_est_curr_mi_grp_imps = 0: 8,449,441
Total records: 22,705,842
Percentage: 37.21%


In [7]:
odo_median = df['cef_est_curr_mi_grp_imps'].median()
print(odo_median) 

40000.0


In [8]:
# Load VIN_BSST parquet file
df_bsst = pd.read_parquet(vin_bsst_path, columns=['VIN', 'BODY_STYLE_SEGMENT_BODY_TYPE'])
print(f"VIN_BSST loaded: {df_bsst.shape}")
df_bsst.head()

VIN_BSST loaded: (26310207, 2)


,VIN,BODY_STYLE_SEGMENT_BODY_TYPE
0,1B3HB48B88D587726,Basic Economy (Car)
1,1C6RR7ST4JS291614,Fullsize Pickup
2,5J6RW2H84LA025942,Sport Utility
3,JN8AS5MV5AW136578,Sport Utility
4,5NMS3CAD0LH161245,Sport Utility


In [9]:
# Merge BODY_STYLE_SEGMENT_BODY_TYPE to df using VIN
df = df.merge(df_bsst, left_on='vin', right_on='VIN', how='left')
print(f"After merge shape: {df.shape}")

After merge shape: (27055821, 12)


In [10]:
# Create BSST_formatted column
# Transform: "Basic Economy (Car)" -> "Basic_Economy_Car_GLM"
df['BSST_formatted'] = (df['BODY_STYLE_SEGMENT_BODY_TYPE']
    .str.replace(' ', '_')
    .str.replace('_(', '_', regex=False)
    .str.replace(')', '', regex=False) + '_GLM')

# Preview the formatting
print("Sample BSST_formatted values:")
print(df[['BODY_STYLE_SEGMENT_BODY_TYPE', 'BSST_formatted']].drop_duplicates().head(10))

Sample BSST_formatted values:
      BODY_STYLE_SEGMENT_BODY_TYPE         BSST_formatted
0              Lower Midsize (Car)  Lower_Midsize_Car_GLM
2              Basic Economy (Car)  Basic_Economy_Car_GLM
10               Entry Level (Car)    Entry_Level_Car_GLM
15              Basic Luxury (Car)   Basic_Luxury_Car_GLM
23             Upper Midsize (Car)  Upper_Midsize_Car_GLM
59              Basic Sporty (Car)   Basic_Sporty_Car_GLM
90             Middle Luxury (Car)  Middle_Luxury_Car_GLM
702                            NaN                    NaN
47743             Fullsize Utility   Fullsize_Utility_GLM
47750                Sport Utility      Sport_Utility_GLM


In [11]:
# Get unique BSST_formatted values
unique_bsst = df['BSST_formatted'].dropna().unique()
print(f"Found {len(unique_bsst)} unique BSST_formatted values:")
for bsst in sorted(unique_bsst):
    print(f"  - {bsst}")

Found 13 unique BSST_formatted values:
  - Basic_Economy_Car_GLM
  - Basic_Luxury_Car_GLM
  - Basic_Sporty_Car_GLM
  - Entry_Level_Car_GLM
  - Fullsize_Utility_GLM
  - Lower_Midsize_Car_GLM
  - Middle_Luxury_Car_GLM
  - Mini_Sport_Utility_GLM
  - Minivan_Passenger_GLM
  - Prestige_Luxury_Car_GLM
  - Prestige_Sporty_Car_GLM
  - Sport_Utility_GLM
  - Upper_Midsize_Car_GLM


## Data Quality Check - Missing Values
Check for zeros, nulls, empty strings for key features before applying defaults

In [12]:
# Check missing values BEFORE applying defaults
print("Model Year from config:", config['model_year'])
missing_summary_before = check_missing_values(df, config['model_year'])

Model Year from config: 2025

DATA QUALITY CHECK - Missing/Zero/Null Values
Total Records: 27,055,821

Feature                                         Zeros     Null/NaN    Empty Str     Total Issue          %
-------------------------------------------------------------------------------------------------------
ODOMETER                                   10,311,982            0            0      10,311,982   38.1137%
geo_pop_density_ntile                             242      163,998            0         163,998    0.6061%
CALC_VEH_AGE (dml_year_imps)                        0            0            0               0    0.0000%
STATE                                               0            0            0               0    0.0000%
BSST                                                0       21,085            0          21,085    0.0779%
-------------------------------------------------------------------------------------------------------

Note: For ODOMETER, zeros are counted as missi

## Apply Default Values
Fill missing values using lookup tables:
- **Odometer**: Lookup by vehicle age (ages > 25 use age 25 default)
- **State**: Lookup by BSST (Body Style Segment Body Type)
- **Population Density Percentile**: Use 50 (median) as default

In [13]:
# Apply all default values
df, fill_counts = apply_all_defaults(df, config)


APPLYING DEFAULT VALUES
✓ Loaded odometer defaults: 26 vehicle age values (0-25)
✓ Loaded state defaults: 16 BSST types

✓ Filled 10,311,982 missing odometer values using vehicle age lookup
  (Ages > 25 used the default for age 25: 124,707 miles)
✓ Created ODOMETER_IMP_FLAG column (10,311,982 records flagged as imputed)
✓ No missing state values to fill
✓ Filled 163,998 missing population density percentile values with default: 50

DEFAULT VALUES APPLIED SUCCESSFULLY
Total values filled: 10,475,980


In [14]:
# Check missing values AFTER applying defaults
missing_summary_after = check_missing_values(df, config['model_year'])


DATA QUALITY CHECK - Missing/Zero/Null Values
Total Records: 27,055,821

Feature                                         Zeros     Null/NaN    Empty Str     Total Issue          %
-------------------------------------------------------------------------------------------------------
ODOMETER                                            0            0            0               0    0.0000%
geo_pop_density_ntile                             242            0            0               0    0.0000%
CALC_VEH_AGE (dml_year_imps)                        0            0            0               0    0.0000%
STATE                                               0            0            0               0    0.0000%
BSST                                                0       21,085            0          21,085    0.0779%
-------------------------------------------------------------------------------------------------------

Note: For ODOMETER, zeros are counted as missing values (to be filled with 

In [15]:
# Calculate CALC_VEH_AGE upfront
df['CALC_VEH_AGE'] = config['model_year'] - df['dml_year_imps']
print(f"CALC_VEH_AGE calculated for {len(df):,} records")
print(f"CALC_VEH_AGE range: {df['CALC_VEH_AGE'].min()} to {df['CALC_VEH_AGE'].max()}")

CALC_VEH_AGE calculated for 27,055,821 records
CALC_VEH_AGE range: 0.0 to 3024.0


## Process Full Data - GLM Predictions
Run GLM predictions for ALL BSST segments and combine results

In [16]:
def prepare_for_glm(df_segment):
    """
    Prepare data for GLM model - creates indicator columns for STATE and MAKE.
    Assumes CALC_VEH_AGE is already calculated in the dataframe.
    
    Parameters:
    -----------
    df_segment : pd.DataFrame
        Data filtered for a specific BSST segment
        
    Returns:
    --------
    pd.DataFrame
        Prepared dataframe with GLM model expected columns
    """
    df_pred = df_segment.copy()
    
    # Map columns to GLM model expected names
    df_pred['ODOMETER'] = df_pred['cef_est_curr_mi_grp_imps']
    # Note: geo_pop_density_ntile is already calculated in the main dataframe
    
    # Create STATE indicator columns
    states = df_pred['st_raw'].unique()
    for state in states:
        if pd.notna(state):
            df_pred[f'STATE_{state}'] = (df_pred['st_raw'] == state)
    
    # Create MAKE indicator columns
    makes = df_pred['dml_make_raw'].unique()
    for make in makes:
        if pd.notna(make):
            df_pred[f'MAKE_{make}'] = (df_pred['dml_make_raw'] == make)
    
    return df_pred

In [17]:
# Process ALL BSST segments
import time

print(f"{'='*80}")
print(f"PROCESSING ALL BSST SEGMENTS")
print(f"{'='*80}")
print(f"Total records: {len(df):,}")
print(f"BSST segments to process: {len(unique_bsst)}")
print()

# Initialize Dep_factor column with NaN
df['Dep_factor'] = pd.NA

start_time = time.time()
total_processed = 0
segments_processed = 0

for bsst in unique_bsst:
    # Load the GLM model for this BSST
    model_path = os.path.join(bsst_glm_path, f"{bsst}.json")
    
    if not os.path.exists(model_path):
        print(f"  ⚠ Model not found: {bsst} - skipping")
        continue
    
    # Get records for this BSST
    mask = df['BSST_formatted'] == bsst
    count = mask.sum()
    
    if count == 0:
        continue
    
    # Load model (suppress output for cleaner logs)
    with open(model_path, 'r') as f:
        glm_model = json.load(f)
    
    # Get segment data, prepare for GLM, and run predictions
    df_segment = df[mask]
    df_pred = prepare_for_glm(df_segment)
    predictions = predict_glm(df_pred, glm_model)
    
    # Add predictions to df_pred for debugging
    df_pred['Dep_factor'] = predictions.values
    
    # Check for problematic records (Dep_factor > 1) and save to debug folder
    mask_problem = df_pred['Dep_factor'] > 1
    if mask_problem.any():
        df_problem = df_pred[mask_problem]
        debug_path = os.path.join(config['debug_folder'], f'Records_{bsst}.parquet')
        os.makedirs(config['debug_folder'], exist_ok=True)
        df_problem.to_parquet(debug_path, index=False)
        print(f"  ⚠ WARNING: {mask_problem.sum():,} records with Dep_factor > 1 saved to: Records_{bsst}.parquet")
    
    # Assign predictions back to main dataframe
    df.loc[mask, 'Dep_factor'] = predictions.values
    
    total_processed += count
    segments_processed += 1
    
    # Progress update every 5 segments
    if segments_processed % 5 == 0 or segments_processed == len(unique_bsst):
        elapsed = time.time() - start_time
        print(f"  Processed {segments_processed}/{len(unique_bsst)} segments ({total_processed:,} records) - {elapsed:.1f}s")

# Calculate veh_value_dep
df['veh_value_dep'] = df['vc_msrp_impa'] * df['Dep_factor']

elapsed = time.time() - start_time
print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"Total segments processed: {segments_processed}")
print(f"Total records with Dep_factor: {df['Dep_factor'].notna().sum():,}")
print(f"Records without Dep_factor (missing BSST): {df['Dep_factor'].isna().sum():,}")
print(f"Total time: {elapsed:.1f} seconds")

PROCESSING ALL BSST SEGMENTS
Total records: 27,055,821
BSST segments to process: 13

  ⚠ WARNING: 39,610 records with Dep_factor > 1 saved to: Records_Lower_Midsize_Car_GLM.parquet
  ⚠ WARNING: 39,239 records with Dep_factor > 1 saved to: Records_Basic_Economy_Car_GLM.parquet
  ⚠ WARNING: 1,800 records with Dep_factor > 1 saved to: Records_Entry_Level_Car_GLM.parquet
  ⚠ WARNING: 302 records with Dep_factor > 1 saved to: Records_Basic_Luxury_Car_GLM.parquet
  ⚠ WARNING: 7,119 records with Dep_factor > 1 saved to: Records_Upper_Midsize_Car_GLM.parquet
  Processed 5/13 segments (20,386,732 records) - 70.0s
  ⚠ WARNING: 14,907 records with Dep_factor > 1 saved to: Records_Basic_Sporty_Car_GLM.parquet
  ⚠ WARNING: 13,086 records with Dep_factor > 1 saved to: Records_Middle_Luxury_Car_GLM.parquet
  ⚠ WARNING: 43,211 records with Dep_factor > 1 saved to: Records_Fullsize_Utility_GLM.parquet
  ⚠ WARNING: 23,222 records with Dep_factor > 1 saved to: Records_Sport_Utility_GLM.parquet
  ⚠ WARNIN

In [18]:
# Summary statistics
print("\nDep_factor Statistics:")
print(df['Dep_factor'].describe())

print("\nveh_value_dep Statistics:")
print(df['veh_value_dep'].describe())


Dep_factor Statistics:
count     2.703474e+07
unique    1.091047e+07
top       4.515342e-01
freq      4.317000e+03
Name: Dep_factor, dtype: float64

veh_value_dep Statistics:
count     27034736.0
unique    12300277.0
top              0.0
freq          4215.0
Name: veh_value_dep, dtype: float64


## Export Results to Parquet
Export selected columns to parquet file

In [19]:
# Validate required columns exist before export
required_columns = ['ODOMETER_IMP_FLAG', 'CALC_VEH_AGE', 'Dep_factor', 'veh_value_dep']
missing_cols = [col for col in required_columns if col not in df.columns]
if missing_cols:
    print(f"ERROR: Missing columns: {missing_cols}")
    print("Make sure you ran these cells in order:")
    print("  1. 'Apply all default values' cell (creates ODOMETER_IMP_FLAG, CALC_VEH_AGE)")
    print("  2. 'Process ALL BSST segments' cell (creates Dep_factor, veh_value_dep)")
    raise KeyError(f"Missing required columns: {missing_cols}")

print("✓ All required columns present in dataframe")
print(f"  Current df columns: {len(df.columns)}")

# Define output columns in order
output_columns = [
    'vin_date',                        # First column
    'vin',
    'BODY_STYLE_SEGMENT_BODY_TYPE',    # BSST
    'cef_est_curr_mi_grp_imps',        # ODOMETER (with defaults applied)
    'ODOMETER_IMP_FLAG',               # Flag for imputed odometer values
    'CALC_VEH_AGE',
    'st_raw',                          # STATE (with defaults applied)
    'vc_msrp_impa',
    'Dep_factor',
    'veh_value_dep'
]

# Create output dataframe with selected columns
df_output = df[output_columns].copy()

# Rename columns for clarity in output
df_output = df_output.rename(columns={
    'cef_est_curr_mi_grp_imps': 'ODOMETER',
    'BODY_STYLE_SEGMENT_BODY_TYPE': 'BSST',
    'st_raw': 'STATE'
})

print("Output columns:")
print(df_output.columns.tolist())
print(f"\nOutput shape: {df_output.shape}")
print(f"\nData types:")
print(df_output.dtypes)

✓ All required columns present in dataframe
  Current df columns: 17
Output columns:
['vin_date', 'vin', 'BSST', 'ODOMETER', 'ODOMETER_IMP_FLAG', 'CALC_VEH_AGE', 'STATE', 'vc_msrp_impa', 'Dep_factor', 'veh_value_dep']

Output shape: (27055821, 10)

Data types:
vin_date              object
vin                   object
BSST                  object
ODOMETER             float64
ODOMETER_IMP_FLAG      int64
CALC_VEH_AGE         float64
STATE                 object
vc_msrp_impa         float64
Dep_factor            object
veh_value_dep         object
dtype: object


In [20]:
# Preview output
df_output.head(10)

,vin_date,vin,BSST,ODOMETER,ODOMETER_IMP_FLAG,CALC_VEH_AGE,STATE,vc_msrp_impa,Dep_factor,veh_value_dep
0,KMHWF35H45A178516_04-26-2022,KMHWF35H45A178516,Lower Midsize (Car),101755.0,1,20.0,OH,19594.0,0.046729,915.600315
1,KMHWF35H75A128502_11-21-2020,KMHWF35H75A128502,Lower Midsize (Car),210000.0,0,20.0,TX,19594.0,0.03005,588.801229
2,KMHD35LE5DU068088_05-03-2020,KMHD35LE5DU068088,Basic Economy (Car),70000.0,0,12.0,CA,19545.0,0.533177,10420.937286
3,KMHD35LE5DU047158_05-30-2021,KMHD35LE5DU047158,Basic Economy (Car),60000.0,0,12.0,GA,18545.0,0.415157,7699.095568
4,KMHD35LE5DU082332_11-17-2019,KMHD35LE5DU082332,Basic Economy (Car),56573.0,1,12.0,NJ,18545.0,0.204333,3789.349329
5,KMHD35LE5DU082332_11-17-2019,KMHD35LE5DU082332,Basic Economy (Car),56573.0,1,12.0,NJ,18545.0,0.204333,3789.349329
6,1HGCP3F81BA000806_05-11-2024,1HGCP3F81BA000806,Lower Midsize (Car),130000.0,0,14.0,NC,29630.0,0.365933,10842.6044
7,KMHD04LB4JU724186_07-24-2020,KMHD04LB4JU724186,Basic Economy (Car),10000.0,0,7.0,CA,22900.0,0.356224,8157.536507
8,KMHWF35H83A750166_08-31-2021,KMHWF35H83A750166,Lower Midsize (Car),113314.0,1,22.0,OH,19114.0,0.083078,1587.957523
9,KMHD35LE9DU147828_02-12-2021,KMHD35LE9DU147828,Basic Economy (Car),200000.0,0,12.0,OH,18545.0,0.28448,5275.677915


In [21]:
# Export to parquet
output_path = '/Users/Mach/dev/aps/data/2026_Dmodel_data/car_with_dep_factor.parquet'

print(f"Exporting to: {output_path}")
df_output.to_parquet(output_path, index=False)

# Verify export
file_size = os.path.getsize(output_path) / (1024 * 1024)  # Size in MB
print(f"\n✓ Export complete!")
print(f"  File size: {file_size:.2f} MB")
print(f"  Records: {len(df_output):,}")
print(f"  Columns: {len(df_output.columns)}")

Exporting to: /Users/Mach/dev/aps/data/2026_Dmodel_data/car_with_dep_factor.parquet

✓ Export complete!
  File size: 1234.14 MB
  Records: 27,055,821
  Columns: 10


In [22]:
# Verify by reading back
df_verify = pd.read_parquet(output_path)
print(f"Verification - read back {len(df_verify):,} records")
print(f"\nODOMETER_IMP_FLAG distribution:")
print(df_verify['ODOMETER_IMP_FLAG'].value_counts())
print(f"\nRecords with imputed odometer: {(df_verify['ODOMETER_IMP_FLAG'] == 1).sum():,}")

Verification - read back 27,055,821 records

ODOMETER_IMP_FLAG distribution:
ODOMETER_IMP_FLAG
0    16743839
1    10311982
Name: count, dtype: int64

Records with imputed odometer: 10,311,982
